# 🔬 DermaFusion-AI — Kaggle Training Notebook
## EVA-02 Large + ConvNeXt V2 | Dual-Branch Fusion | SOTA 2026
### ⚙️ Before Running:
1. Datasets already added ✅
2. **Accelerator → GPU T4 x2** ✅
3. **Persistence → Files only** ✅
4. Run cells top to bottom — Steps 5+6 are combined to auto-start classifier after segmentation

## Step 0: Inspect Input Paths

In [ ]:
import os

def explore(path, depth=0, max_depth=2):
    if depth > max_depth or not os.path.exists(path):
        return
    for item in sorted(os.listdir(path)):
        full = os.path.join(path, item)
        kind = '📁' if os.path.isdir(full) else '📄'
        print('  ' * depth + f'{kind} {item}')
        if os.path.isdir(full):
            explore(full, depth + 1, max_depth)

explore('/kaggle/input', max_depth=2)

## Step 1: Clone / Pull Repo

In [ ]:
import os
if not os.path.exists('/kaggle/working/DermaFusion-AI'):
    !git clone https://github.com/ai-with-abdullah/DermaFusion-AI.git /kaggle/working/DermaFusion-AI
else:
    !cd /kaggle/working/DermaFusion-AI && git pull

%cd /kaggle/working/DermaFusion-AI
print('Working directory:', os.getcwd())

## Step 2: Install Dependencies

In [ ]:
%pip install -q timm>=0.9.12 albumentations>=1.3.1 opencv-python-headless scikit-learn scipy tqdm
print('✅ Dependencies installed')

## Step 3: Verify GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'✅ GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB')
else:
    print('❌ No GPU — Settings > Accelerator > GPU T4 x2')

## Step 4: Link All Datasets → data/ folder

In [ ]:
import os

data_dir = '/kaggle/working/DermaFusion-AI/data'
os.makedirs(data_dir, exist_ok=True)

search_roots = ['/kaggle/input/competitions']
datasets_dir = '/kaggle/input/datasets'
if os.path.exists(datasets_dir):
    for username in os.listdir(datasets_dir):
        user_path = os.path.join(datasets_dir, username)
        if os.path.isdir(user_path):
            search_roots.append(user_path)
search_roots.append('/kaggle/input')

def find_folder(keywords):
    # Search up to 3 levels deep in /kaggle/input
    paths_to_check = ['/kaggle/input']
    checked_paths = set()
    for depth in range(3):
        next_paths = []
        for p in paths_to_check:
            if not os.path.exists(p) or p in checked_paths:
                continue
            checked_paths.add(p)
            try:
                for item in os.listdir(p):
                    full = os.path.join(p, item)
                    if os.path.isdir(full):
                        if any(kw in item.lower() for kw in keywords):
                            return full
                        next_paths.append(full)
            except Exception:
                pass
        paths_to_check = next_paths
    return None

def link(src, dst_name, label):
    dst = os.path.join(data_dir, dst_name)
    if os.path.exists(dst):
        print(f'✅ {label}: already linked')
        return
    if src:
        os.symlink(src, dst)
        print(f'✅ {label}: linked from {src}')
    else:
        print(f'❌ {label}: NOT FOUND')

link(find_folder(['ham10000', 'skin-cancer-mnist']),               'ham10000',  'HAM10000')
link(find_folder(['isic-2019', 'isic2019', 'skin-lesion-images']), 'isic_2019', 'ISIC 2019')
link(find_folder(['siim-isic', 'melanoma-classification']),        'isic_2020', 'ISIC 2020')
link(find_folder(['isic-2024', 'skin-cancer-detection-3d']),       'isic_2024', 'ISIC 2024')
link(find_folder(['pad-ufes', 'padufes']),                         'pad_ufes',  'PAD-UFES-20')
link(find_folder(['ddi', 'diverse-dermatology']),                  'ddi',       'Stanford DDI')

derm7pt_src = find_folder(['derm7pt', 'release-v0', 'release_v0'])
if derm7pt_src:
    dst = '/kaggle/working/DermaFusion-AI/release_v0'
    if not os.path.exists(dst):
        os.symlink(derm7pt_src, dst)
        print(f'✅ DERM7PT: linked to {dst}')

print('\n📁 data/ contents:', os.listdir(data_dir))

## Step 5: Phase 1 — Segmentation Training (Swin-UNet)
Trains the Swin-Transformer U-Net to perform lesion masking. Saves checkpoints after each epoch to `/kaggle/working/DermaFusion-AI/outputs/weights/resume_seg_checkpoint.pth`. If the session stops, re-running this cell will automatically resume from the last epoch.

In [ ]:
%cd /kaggle/working/DermaFusion-AI
print('=' * 60)
print('🚀 PHASE 1 — Segmentation Training (25 epochs)')
print('   Swin-Tiny UNet | SEG_BATCH_SIZE=8 | 2× T4 GPUs')
print('=' * 60)
!PYTHONPATH=. python train_segmentation.py 2>&1 | tee /kaggle/working/train_segmentation.log
print('\n✅ Phase 1 complete!')

## Step 6: Phase 2 — Classifier Training (EVA-02 + ConvNeXt V2)
Trains the dual-branch fusion classification model. Automatically loads the trained Swin-UNet weights. Saves checkpoints after each epoch to `/kaggle/working/DermaFusion-AI/outputs/weights/resume_checkpoint.pth`. Re-running this cell will automatically resume from the last epoch.

In [ ]:
%cd /kaggle/working/DermaFusion-AI
print('=' * 60)
print('🚀 PHASE 2 — Classifier Training (25 epochs)')
print('   EVA-02 Large + ConvNeXt V2 | BATCH_SIZE=2 | 2× T4 GPUs')
print('=' * 60)
!PYTHONPATH=. python train_classifier.py 2>&1 | tee /kaggle/working/train_classifier.log
print('\n✅ Phase 2 complete!')

## Step 7: Export Weights, Results and Plots to Output Tab
Copies the trained model weights, the **Ablation Study results CSV**, the **Evaluation Dashboard**, and all individual plots to `/kaggle/working/` so they appear in the Kaggle Output panel on the right for easy download.

In [ ]:
import shutil, os

# 1. Export Weights
weights_src = '/kaggle/working/DermaFusion-AI/outputs/weights/'
if os.path.exists(weights_src):
    for f in os.listdir(weights_src):
        if f.endswith('.pth'):
            dst = f'/kaggle/working/{f}'
            shutil.copy(os.path.join(weights_src, f), dst)
            print(f'✅ Weights: {f}  ({os.path.getsize(dst)/1e6:.0f} MB)')
else:
    print('⚠️ No weights folder found (yet)')

# 2. Export Ablation CSV Results
csv_path = '/kaggle/working/DermaFusion-AI/outputs/ablation_study_results.csv'
if os.path.exists(csv_path):
    shutil.copy(csv_path, '/kaggle/working/ablation_study_results.csv')
    print('✅ Results: ablation_study_results.csv')

# 3. Export Plots and Dashboard
plots_src = '/kaggle/working/DermaFusion-AI/outputs/plots/'
if os.path.exists(plots_src):
    for f in os.listdir(plots_src):
        if f.endswith(('.png', '.jpg', '.jpeg')):
            shutil.copy(os.path.join(plots_src, f), f'/kaggle/working/{f}')
            print(f'✅ Plot: {f}')

print('\n📦 Files exported to /kaggle/working/ and ready for download!')

## Step 8: Full Evaluation & Ablation Study

### 📅 Exam Preparation Mode (Run Ablation Study NOW):
If you want to run the ablation study **now** using your existing weights (to avoid waiting 24+ hours for training during exams):
1. Skip Step 5 and Step 6 cells.
2. Copy your existing model weights to `/kaggle/working/DermaFusion-AI/outputs/weights/best_unet.pth` and `best_dual_branch_fusion.pth` (or let the script load them from your active persistent inputs).
3. Run this cell below. It will run the full evaluation, generate separate confusion matrices, run the Ablation Study configurations 1-6, and generate the final dashboard.
4. Run **Step 7** afterwards to make sure the plots and CSV are copied to the Output tab for downloading.

### 🏁 Training from Scratch (After Exams in 2 weeks):
1. Clear all old weights: `!rm -rf /kaggle/working/DermaFusion-AI/outputs/weights/*` in a cell.
2. Run Step 5 (Segmentation) and Step 6 (Classifier) to train fresh from zero.
3. Run this cell to evaluate your new final models.

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python evaluate.py 2>&1 | tee /kaggle/working/evaluate.log
print('\n✅ Evaluation complete!')

## Step 9: Advanced Paper Evaluations
Runs calibration, bootstrap confidence intervals, McNemar significance tests, inference benchmarking, 5-fold CV, and external smartphone/dermoscopy evaluations (PAD-UFES-20 and DERM7PT).

In [ ]:
%cd /kaggle/working/DermaFusion-AI

print('=== 1. Probability Calibration (Temperature Scaling) ===')
!PYTHONPATH=. python -m evaluation.run_temperature_scaling

print('\n=== 2. Statistical Significance (McNemar\'s Test) ===')
!PYTHONPATH=. python -m evaluation.run_statistical_tests

print('\n=== 3. Statistical Confidence Intervals (Bootstrap) ===')
!PYTHONPATH=. python -m evaluation.run_confidence_intervals

print('\n=== 4. Inference Latency & Speed Benchmark ===')
!PYTHONPATH=. python -m evaluation.run_inference_benchmark

print('\n=== 5. 5-Fold Cross-Validation (EVA-02 Small) ===')
!PYTHONPATH=. python -m evaluation.run_cross_validation

print('\n=== 6. Smartphone External Test (PAD-UFES-20) ===')
!PYTHONPATH=. python test_padufes.py

print('\n=== 7. Smartphone Head Fine-Tuning ===')
!PYTHONPATH=. python finetune_padufes.py

print('\n=== 8. DERM7PT External Test (4 combinations) ===')
!PYTHONPATH=. python test_both_weights.py

print('\n✅ All advanced paper evaluations are complete!')